# Steam analysis: shared data setup
Loads and cleans the Steam catalog and builds the dataframes used by every analysis notebook. It is executed by the other notebooks with `%run ./setup_data` and displays nothing on purpose (lighter published notebooks). The exploration of the raw data and the justification of each cleaning step are in `00_data_exploration`.

| Dataframe | Grain | Content |
|---|---|---|
| `games_df` | 1 row per game | cleaned columns (prices, dates, owners, reviews, ages, platforms) |
| `genres_df` | 1 row per (game, genre) | `genre` exploded |
| `languages_df` | 1 row per (game, language) | `language` exploded |
| `platforms_df` | 1 row per (game, platform) | Windows / Mac / Linux |
| `tags_df` | 1 row per (game, tag) | community tags and their vote count |

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

DATA_PATH = "s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json"
# Fallback if the S3 bucket is not reachable from your workspace: download
# https://full-stack-bigdata-datasets.s3.eu-west-3.amazonaws.com/Big_Data/Project_Steam/steam_game_output.json
# upload it in Catalog > workspace > default > Create volume "steam" > Upload, then use:
# DATA_PATH = "/Volumes/workspace/default/steam/steam_game_output.json"
raw_df = spark.read.json(DATA_PATH)
flat_df = raw_df.select("data.*")

### Cleaning and feature engineering
- **prices** are strings in cents: converted to USD (`try_cast` returns null instead of failing on odd values, which matters on Spark 4 / serverless where ANSI mode is on).
- **release_date** comes as `yyyy/M/d` or only `yyyy/M`: parsed to a date, `release_year` and `release_month` extracted.
- **owners** is a range (`"20,000 .. 50,000"`): lower bound, upper bound and midpoint kept. It is an estimate of sales volume (SteamSpy).
- **reviews**: total, positive ratio and the Wilson lower bound of the positive ratio (ranks games by rating while penalising small review counts).
- **required_age**: first number extracted (`"MA 15+"` becomes 15).
- **estimated_revenue_usd** = owners midpoint x current price: a rough order of magnitude, not actual sales (prices change, discounts, regional pricing).
- Only `type == "game"` is kept; duplicated `appid` are removed.

In [0]:
owners_bounds = F.split(F.regexp_replace("owners", ",", ""), r" \.\. ")
pos, neg = F.col("positive"), F.col("negative")
n = pos + neg
z = 1.96  # Wilson score lower bound, 95%
wilson = (pos / n + z * z / (2 * n) - z * F.sqrt((pos / n) * (1 - pos / n) / n + z * z / (4 * n * n))) / (1 + z * z / n)

parsed_date = F.coalesce(
    F.to_date(F.expr("try_to_timestamp(release_date, 'yyyy/M/d')")),
    F.to_date(F.expr("try_to_timestamp(release_date, 'yyyy/M')")),
)

games_df = (flat_df
    .filter(F.col("type") == "game")
    .dropDuplicates(["appid"])
    .withColumn("publisher", F.when(F.trim("publisher") == "", None).otherwise(F.trim("publisher")))
    .withColumn("price_usd", F.expr("try_cast(price AS INT)") / 100)
    .withColumn("initial_price_usd", F.expr("try_cast(initialprice AS INT)") / 100)
    .withColumn("discount_pct", F.coalesce(F.expr("try_cast(discount AS INT)"), F.lit(0)))
    .withColumn("is_free", F.col("price_usd") == 0)
    .withColumn("release_date_clean", parsed_date)
    .withColumn("release_year", F.year("release_date_clean"))
    .withColumn("release_month", F.date_trunc("month", "release_date_clean").cast("date"))
    .withColumn("_owners", owners_bounds)
    .withColumn("owners_min", F.expr("try_cast(_owners[0] AS BIGINT)"))
    .withColumn("owners_max", F.expr("try_cast(_owners[1] AS BIGINT)"))
    .withColumn("owners_mid", (F.col("owners_min") + F.col("owners_max")) / 2)
    .withColumn("total_reviews", n)
    .withColumn("positive_ratio", F.when(n > 0, pos / n))
    .withColumn("wilson_score", F.when(n > 0, wilson))
    .withColumn("_age", F.regexp_extract(F.coalesce("required_age", F.lit("")), r"(\d+)", 1))
    .withColumn("required_age_int", F.coalesce(F.expr("try_cast(_age AS INT)"), F.lit(0)))
    .withColumn("genres", F.when(F.trim("genre") != "", F.split("genre", r",\s*")).otherwise(F.array().cast("array<string>")))
    .withColumn("languages_clean", F.regexp_replace(F.coalesce("languages", F.lit("")), r"\[b\]\*\[/b\]|\[/?b\]|\*", ""))
    .withColumn("languages_list", F.when(F.trim("languages_clean") != "", F.split("languages_clean", r",\s*")).otherwise(F.array().cast("array<string>")))
    .withColumn("n_genres", F.size("genres"))
    .withColumn("n_languages", F.size("languages_list"))
    .withColumn("n_platforms", F.col("platforms.windows").cast("int") + F.col("platforms.mac").cast("int") + F.col("platforms.linux").cast("int"))
    .withColumn("estimated_revenue_usd", F.col("owners_mid") * F.col("price_usd"))
    .drop("short_description", "header_image", "website", "_owners", "_age")
)

In [0]:
genres_df = games_df.select("appid", "name", "publisher", "price_usd", "owners_mid", "estimated_revenue_usd", "positive",
                            "negative", "total_reviews", "positive_ratio", "ccu", "platforms", F.explode("genres").alias("genre"))
languages_df = games_df.select("appid", "name", F.explode("languages_list").alias("language")).withColumn("language", F.trim("language"))
platforms_df = (games_df
    .select("appid", "name", F.explode(F.array(
        F.when(F.col("platforms.windows"), F.lit("Windows")),
        F.when(F.col("platforms.mac"), F.lit("Mac")),
        F.when(F.col("platforms.linux"), F.lit("Linux")))).alias("platform"))
    .filter(F.col("platform").isNotNull()))
tag_map = F.from_json(F.to_json(F.col("data.tags")), "map<string,bigint>")
tags_df = (raw_df.filter(F.col("data.type") == "game")
    .select(F.col("data.appid").alias("appid"), F.explode(tag_map).alias("tag", "votes")))